# alternating soundsource analysis

In [1]:
from scipy.io import loadmat
from scipy.signal import butter, filtfilt
import numpy as np
import pandas as pd
import polars as pl
import sqlite3
from pathlib import Path
import matplotlib.pyplot as plt
from workbench.data.preprocess import TriggRasterPY, combTableCreate, processTableRow, expand_dict_columns
import h5py
import tables as tb

In [2]:
conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")
sql = """
SELECT * FROM Recordings WHERE (Animal_Id, Cell_Id) IN (
SELECT Animal_Id, Cell_Id FROM Recordings WHERE Condition IN ("Baseline", "soso") GROUP BY Animal_Id, Cell_Id 
HAVING COUNT(DISTINCT Condition) >= 2) 
AND Condition IN ("Baseline","soso") AND use = 1 AND Folders_generated = 1
"""

datatable= pd.read_sql_query(sql, conn)
conn.close()

In [24]:
try:
    comb_table = pl.read_parquet(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\comb_tables\soso_comb.parquet")
except:
    print('comb_table doesnt exist, start creation')
    comb_table = combTableCreate(datatable, "", "")
    comb_df = pd.DataFrame(comb_table)
    dict_list = [ 'pupil_psth', 'whisk_psth']
    comb_df = expand_dict_columns(
        comb_df,
        dict_columns=dict_list,
        flatten_2d=False
    )
    comb_table = pl.from_dataframe(comb_df)
    comb_table.write_parquet(r"Z:\lab share\Data\Florian\comb_tables\soso_comb.parquet", use_pyarrow=True)

comb_table doesnt exist, start creation
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 4, Baseline being processed
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 4, soso being processed
----------------------------------------------------------------------------------------------------
\\172.25.250.112\burgalossi\lab share\Data\Florian\ADN\FH8Soso\analysis\Data4\soso\exp_data.mat
----------------------------------------------------------------------------------------------------
Animal_Id FH8soso; Cell_Id 6, Baseline being processed
----------------------------------------------------------------------------------------------------
-------------------------------------------------------------------------------------------------

In [26]:
# clean an reshape the dataframe
conditions = comb_table['Condition'].unique().to_list()

comb_a = (
    comb_table.filter(pl.col("Condition") == conditions[0]).drop("Condition")
)
comb_b = (
    comb_table.filter(pl.col("Condition") == conditions[1]).drop("Condition")
)

comb_joined = comb_a.join(
    comb_b,
    on=['Animal_Id', 'Cell_Id'],
    how="inner"
)

comb_joined = comb_joined[[s.name for s in comb_joined if not (s.null_count() == comb_joined.height)]]